# MultiDevice Data Usage 

This notebook explains how to use this repo's multidevice etl pipeline, and then how to use the processed data to view and retrieve for each device (`device_type`, `device_id`), its dataframes (`df`); and each df's variables (AKA columns). 

## Load All Data

The function `load_data()` automatically triggers the etl (extract, transform, load) pipeline to prepare all available device data for analysis. 
This is the one-liner command to load everything. 

**Note**: This pipeline is currently extracts from a mounted Proton Drive folder, it can be upgraded to API calls per `device_id` instead.

In [ ]:
import sys
sys.path.insert(0, "..") # run at repo root

from src.etl.load import load_data

MOUNT_PATH = "/home/yul/mnt/proton-data"

data = load_data(MOUNT_PATH)

In [ ]:
# See which device types are available 
list_device_types = data.keys() 
print(list_device_types)

# See available device IDs, per type
list_device_ids = data["Atmotube"].keys()
print(list_device_ids)

In [ ]:
# Alternatively, this is how to extract the raw, unprocessed data
from src.etl.extract import extract_raw_data

raw = extract_raw_data("/home/yul/mnt/proton-data")
raw["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]  # example

## View Available Devices

For each device type (`device_type`), the actual devices are indexed according to its device ID (`device_id`). 

The function `skim()` quickly shows the available dataframes (`df`) per actual device (`device_id`).

It returns, per each available device, its shape (columns nos. and rows nos.) and range (start datetime to end datetime).

In [ ]:
from src.utils import skim

# a. View ALL of the loaded available data, devices, and device types
# skim(data) 

# b. View loaded available data of a device type
# skim(data, "Atmotube") 

# c. View loaded avail data of a specific device ID
skim(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026") # add "pm" after device_id to look for a specific df

# (equivalent manual version, for comparison:)
# display_loaded_data({"Atmotube": {"C3CBE16AE294_01-May-2026_12-Jun-2026": data["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]}})

The same function `skim()` can also be used more specifically by quickly insepecting one dataframe (`df`). 
It returns, of the specified df, every available variable's (AKA columns) row length and data type (aka dtype)

In [ ]:
skim(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", "pm")

In [ ]:
skim(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", "pm", "pm10_ugm3_atm")

### Structural Relationship of the Data 

Below represents the structural hierarchical relationship between the data based on device type, devices, and each device's dataframes and their variables. 

They are all retrievable as a dictionary of each other. 

In [ ]:
import json

# Structure of the entire data hierarchy
print("Available Device Types:")
print(json.dumps({k: type(v).__name__ for k, v in data.items()}, indent=2))

print("\nDevices in Atmotube:")
print(json.dumps({k: type(v).__name__ for k, v in data["Atmotube"].items()}, indent=2))

from src.utils import get
device = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026")

print("\nStructure of one Device:")
print(json.dumps({k: type(v).__name__ for k, v in device.items()}, indent=2))


## Retrieve Device Data

Since any device and its contents are all kept as nested dictionaries. 

The function `get()` can navigate and retrieve a device, its dataframes, and its variables (columns).

It will return  either the nested dict (for multiple devices) or flattened dict (for one device) — the returned structure is exactly how the data is being kept. 

### 1. For One or More Devices



In [ ]:
from src.utils import get, use

all_devices = get(data) # This will literally get all the loaded available data, 
# print(all_devices)    # showing its implemented structure as one large nested dict

# a. Retrieve all devices of a one type (nested dict)
atmotube_all_devices = get(data, "Atmotube")
# print(atmotube_devices)

# b. Retrieve one device ID (flattened dict)
atmotube_device = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026") 
# print(atmotube_device)

# b. Retrieve two or more device ID (flattened dict)
atmotube_two_devices = get(
    data, "Atmotube", 
    ["C7A595F09965_01-May-2026_12-Jun-2026", 
    "DB7A737B8CA0_01-May-2026_12-Jun-2026"], 
    df_key="pm")  
print(atmotube_two_devices)


# (equivalent manual version, for comparison:)
# device1_pm_df = data["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]["data"]

### 2. For One or More DataFrames

In [ ]:
from src.utils import get

# a. Retrieve one dataframe 
atmotube_device_pm = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", ["pm"])
# print(atmotube_device_pm)

# b. Retrieve two or more dataframes 
atmotube_device_weatherpm = get(
    data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026",
    ["pm", "weather"]
)

print(atmotube_device_weatherpm.keys())  # Check what the dataframes are
print(atmotube_device_weatherpm) 

# (equivalent manual version, for comparison:)
# device1_pm_df = data["Atmotube"]["C3CBE16AE294_01-May-2026_12-Jun-2026"]["data"]["pm"]["df"]

In [ ]:

pm_df = atmotube_device_weatherpm["pm"]
weather_df = atmotube_device_weatherpm["weather"]

use(atmotube_device_weatherpm)

In [ ]:
from src.utils import get

# d. Retrieve a specific variable 
atmotube_device_two_pms = get(data, "Atmotube", "C7A595F09965_01-May-2026_12-Jun-2026", ["pm"], ["pm2_5_ugm3_atm", "pm10_ugm3_atm"])

print(atmotube_device_two_pms)

The function `use()` can return a usable dataframe ready for analysis. 

In [ ]:
from src.utils import use

use(atmotube_device) # merges all dataframes in one device, returns a ready-to-use table

use(atmotube_device)

### Select for DataFrames or Variables

In [ ]:
from src.utils import get

